### Pratik Pujari | 23102B0008 | CMPN B

### CART in Regression vs Classification

**Split criterion:**
- Regression tree → MSE / variance reduction  
- Classification tree → Gini index or Entropy  

**Prediction at leaf:**
- Regression → mean value of target (average LPA)  
- Classification → majority class or probability (Placed / Not Placed)  

**Why prefer CART over linear/logistic?**
CART can capture non-linear relationships and feature interactions (skills, branch, internships), which linear/logistic models may not handle well.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# Load dataset (uploaded file)
df = pd.read_csv("bank.csv", sep=";")

# Encode categorical variables
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

# Features & target
X = df.drop("y", axis=1)
y = df["y"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Train Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict probabilities
probs = model.predict_proba(X_test)[:,1]

# Threshold = 0.5
threshold = 0.5
y_pred = (probs >= threshold).astype(int)

# Metrics
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, probs)

# Sensitivity & Specificity
tn, fp, fn, tp = cm.ravel()
sensitivity = tp/(tp+fn)
specificity = tn/(tn+fp)

print("Confusion Matrix:\n", cm)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Sensitivity:", sensitivity)
print("Specificity:", specificity)
print("ROC-AUC:", roc_auc)

# -------- Threshold analysis --------

fpr, tpr, thresholds = roc_curve(y_test, probs)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print("\nOptimal Threshold:", optimal_threshold)

# Predictions with optimal threshold
y_pred_opt = (probs >= optimal_threshold).astype(int)

# Metrics at optimal threshold
precision_opt = precision_score(y_test, y_pred_opt)
recall_opt = recall_score(y_test, y_pred_opt)
f1_opt = f1_score(y_test, y_pred_opt)

print("\nMetrics at optimal threshold")
print("Precision:", precision_opt)
print("Recall:", recall_opt)
print("F1:", f1_opt)

# Save probabilities file
output = pd.DataFrame({
    "RecordId": range(len(probs)),
    "Probability(yes)": probs,
    "PredictedLabel": y_pred
})

output.to_csv("probabilities.csv", index=False)

print("\nprobabilities.csv saved successfully")


Confusion Matrix:
 [[983  23]
 [104  21]]
Precision: 0.4772727272727273
Recall: 0.168
F1 Score: 0.2485207100591716
Sensitivity: 0.168
Specificity: 0.9771371769383698
ROC-AUC: 0.8637296222664016

Optimal Threshold: 0.12009636513077573

Metrics at optimal threshold
Precision: 0.30029154518950435
Recall: 0.824
F1: 0.44017094017094016

probabilities.csv saved successfully


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Why ROC curve is useful
ROC curve shows how well the model separates classes at different thresholds.  
It helps evaluate performance beyond accuracy and is useful for comparing models, especially when data is imbalanced.

### What changes when threshold changes (Precision–Recall trade-off)
- Lower threshold → Recall increases, Precision decreases  
- Higher threshold → Precision increases, Recall decreases  
Changing threshold helps balance false positives and false negatives based on business needs.
